# Week 5 — Digging Deeper & Making Figures

We have our result: behavior improves neural prediction. This week we ask the follow-up questions a scientist would ask, and make clean figures for the write-up:

1. How is the improvement spread across neurons?
2. Do the arousal-sensitive neurons really benefit most?
3. Do behavior-helped neurons cluster anywhere in the cortex?
4. What does one dramatic example look like?

## 1. Setup — rebuild the models and load neuron locations

In [ ]:
import os, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

FILENAME = "static21067-10-18-GrayImageNet-94c6ff995dac583098847cfecd43e7b6"

if not os.path.exists("week3_prepared.npz"):
    print("Run Week 3 first, or the rebuild cell in Week 4, to create week3_prepared.npz")

d = np.load("week3_prepared.npz", allow_pickle=True)
S, R, B, tiers = d["stim_features"], d["responses"], d["behavior"], d["tiers"]
is_train = (tiers == "train"); is_test = (tiers == "validation") | (tiers == "test")

Bs = StandardScaler().fit(B[is_train]).transform(B)

def corr(P, A):
    P = P - P.mean(0); A = A - A.mean(0)
    return (P*A).sum(0) / (np.sqrt((P**2).sum(0)*(A**2).sum(0)) + 1e-12)

# fit both models, KEEP the predictions (we'll plot them)
mS  = Ridge(alpha=1000.).fit(S[is_train], R[is_train]);            predS  = mS.predict(S[is_test])
XSB = np.hstack([S, Bs])
mSB = Ridge(alpha=1000.).fit(XSB[is_train], R[is_train]);          predSB = mSB.predict(XSB[is_test])

corr_stim = corr(predS,  R[is_test])
corr_both = corr(predSB, R[is_test])
improvement = corr_both - corr_stim

# arousal modulation of each neuron (from Week 2)
xc = B[:,0]-B[:,0].mean(); Rc = R - R.mean(0)
pupil_mod = (Rc*xc[:,None]).sum(0)/(np.sqrt((Rc**2).sum(0)*(xc**2).sum())+1e-12)

# neuron locations in the cortex (x, y, depth), from the dataset metadata
coords = np.load(f"{FILENAME}/meta/neurons/cell_motor_coordinates.npy", allow_pickle=True)
print("Ready. mean improvement = +%.3f" % improvement.mean())

## 2. How is the improvement spread out?

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(improvement, bins=60, color="#5aa76a")
plt.axvline(0, color="k", lw=0.8)
plt.axvline(improvement.mean(), color="red", ls="--", label=f"mean +{improvement.mean():.3f}")
plt.xlabel("improvement from behavior"); plt.ylabel("number of neurons")
plt.title(f"{100*np.mean(improvement>0):.0f}% of neurons improved"); plt.legend()
plt.show()
print("mean %.3f | median %.3f | biggest single-neuron gain %.3f" % (
      improvement.mean(), np.median(improvement), improvement.max()))

The mean is bigger than the median — most neurons gain a little, and a **tail of neurons gains a lot**. Who are those big winners?

## 3. The big winners are the arousal-sensitive neurons

In [ ]:
plt.figure(figsize=(6,4.5))
plt.scatter(pupil_mod, improvement, alpha=0.12, s=8, color="#333")
# average trend line
bins = np.linspace(pupil_mod.min(), pupil_mod.max(), 15); idx = np.digitize(pupil_mod, bins)
bx = [pupil_mod[idx==i].mean() for i in range(1,len(bins))]
by = [improvement[idx==i].mean() for i in range(1,len(bins))]
plt.plot(bx, by, color="red", lw=2, label="average")
plt.axhline(0, color="k", lw=0.6); plt.axvline(0, color="k", lw=0.6)
plt.xlabel("neuron's arousal modulation (corr with pupil)")
plt.ylabel("improvement from behavior")
plt.title("Arousal-sensitive neurons benefit most"); plt.legend(); plt.show()

## 4. Does the effect cluster in the cortex?

Each neuron has a physical location. If we color each neuron by how much behavior helped it, do the helped neurons bunch together — or are they mixed throughout?

In [ ]:
plt.figure(figsize=(6,5))
sc = plt.scatter(coords[:,0], coords[:,1], c=improvement, s=6,
                 cmap="viridis", vmin=0, vmax=np.quantile(improvement, 0.98))
plt.xlabel("cortical x (microns)"); plt.ylabel("cortical y (microns)")
plt.title("Improvement across the cortical surface")
plt.colorbar(sc, label="improvement"); plt.gca().set_aspect("equal"); plt.show()
print("correlation between improvement and cortical depth: %.3f (near 0 = no depth pattern)"
      % np.corrcoef(improvement, coords[:,2])[0,1])

The helped neurons are **mixed throughout** — no obvious spatial clustering. That's a real (if quiet) finding: arousal modulation is a property spread across the population, not confined to one patch.

## 5. One dramatic example

Let's look at the neuron that improved the most, comparing its predictions before and after adding behavior.

In [ ]:
best = int(np.argmax(improvement * (corr_both > 0.2)))
fig, axes = plt.subplots(1, 2, figsize=(10,4.3), sharex=True, sharey=True)
for ax, pred, c, ttl in [(axes[0], predS, corr_stim, "stimulus only"),
                          (axes[1], predSB, corr_both, "stimulus + behavior")]:
    ax.scatter(R[is_test][:,best], pred[:,best], alpha=0.3, s=12, color="#444")
    lims=[R[is_test][:,best].min(), R[is_test][:,best].max()]; ax.plot(lims,lims,"r--",lw=0.8)
    ax.set_title(f"{ttl}  (corr={c[best]:.2f})"); ax.set_xlabel("actual response")
axes[0].set_ylabel("predicted response")
fig.suptitle(f"Neuron {best}: behavior improves prediction by +{improvement[best]:.2f}")
plt.tight_layout(); plt.show()

## 6. Save the figures for the report

(Optional — re-runs the plots and saves them as image files you can download.)

In [ ]:
# The figures above are also saved in the GitHub repo under results/.
# To save your own copies from this session, right-click any plot and 'Save image',
# or use plt.savefig('myfigure.png', dpi=150) after any plot.
print("Analysis complete. See docs/REPORT.md in the repo for the written summary.")

## Recap

- Improvement is **right-skewed**: most neurons gain a little, a tail gains a lot.
- The big winners are the **arousal-sensitive** neurons — confirming the story.
- The effect is **spread across the cortex**, not clustered.
- One neuron went from 0.05 → 0.40 correlation just by adding behavior.

### Next: the write-up
Week 6 turns all of this into a short report (`docs/REPORT.md`) — the document you can attach to your application.

You've now done exploration, modeling, AND analysis. That's a full research cycle. 🧠📊